# Semantic Segmentation Training Demo - Real VOC 2012

This notebook demonstrates real training on Pascal VOC 2012 segmentation dataset.
All data, metrics, and visualizations are real - no synthetic placeholders.

## Steps
1. Load real VOC 2012 dataset
2. Visualize samples
3. Create U-Net ResNet18 model
4. Train (fast mode: 80 samples, 128px, 3 epochs for CPU)
5. Evaluate with real mIoU/Dice
6. Visualize predictions

In [ ]:
import sys
sys.path.append('../src')
from data_loader import get_dataloaders
from models import get_model, CombinedLoss
from train import train_model, calculate_iou, calculate_dice
import torch
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# Load real data - Pascal VOC 2012
train_loader, val_loader = get_dataloaders(batch_size=4, img_size=128, num_samples=80, root='../data')
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

# Visualize one batch
images, masks = next(iter(train_loader))
print(f"Images: {images.shape}, Masks: {masks.shape}")
print(f"Mask unique values: {torch.unique(masks)}")

# Denormalize for visualization
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
img = images[0] * std + mean
img = torch.clamp(img, 0, 1).permute(1,2,0).numpy()

fig, ax = plt.subplots(1,2, figsize=(8,4))
ax[0].imshow(img)
ax[0].set_title('Real VOC Image')
ax[0].axis('off')
ax[1].imshow(masks[0].numpy(), cmap='gray')
ax[1].set_title('Real Binary Mask (FG vs BG)')
ax[1].axis('off')
plt.show()

In [ ]:
# Create model - U-Net ResNet18 pretrained
model = get_model(model_name='unet', num_classes=2, encoder='resnet18')
model = model.to(device)
print(model)

# Count params
params = sum(p.numel() for p in model.parameters())
print(f"Total params: {params/1e6:.2f}M")

In [ ]:
# Train - real training loop
# Fast mode for CPU: 3 epochs, 80 samples, 128px
# This will produce real training_curves_unet_real.png
history = train_model(
    model_name='unet',
    encoder='resnet18',
    epochs=3,
    batch_size=4,
    img_size=128,
    num_samples=80,
    lr=1e-4
)
print("Training done - Best mIoU:", max(history['val_miou']))

In [ ]:
# Load best model and evaluate on real val set
from evaluate import evaluate_model
metrics = evaluate_model(model_name='unet', encoder='resnet18', img_size=128, num_samples=80)
print(f"Real validation - mIoU: {metrics['miou']:.4f}, Dice: {metrics['dice']:.4f}, PixelAcc: {metrics['pixel_acc']:.4f}")

# Show real prediction images
from pathlib import Path
demo_images = list(Path('../demo').glob('real_pred*.jpg'))[:3]
for img_path in demo_images:
    img = plt.imread(img_path)
    plt.figure(figsize=(10,4))
    plt.imshow(img)
    plt.axis('off')
    plt.title(img_path.name)
    plt.show()

In [ ]:
# Inference test - real API
from inference import SegmentationInference
from PIL import Image
import numpy as np

inference = SegmentationInference(model_name='unet', encoder='resnet18')

# Take a real val image
images, masks = next(iter(val_loader))
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
img_tensor = images[0] * std + mean
img_tensor = torch.clamp(img_tensor, 0, 1)
img_np = (img_tensor.permute(1,2,0).numpy()*255).astype(np.uint8)

pred = inference.predict(img_np)
print(f"Predicted mask shape: {pred.shape}, unique: {np.unique(pred)}")

original, mask_resized, overlay = inference.predict_with_overlay(img_np, alpha=0.5)
fig, ax = plt.subplots(1,3, figsize=(12,4))
ax[0].imshow(original)
ax[0].set_title('Original')
ax[0].axis('off')
ax[1].imshow(mask_resized, cmap='gray')
ax[1].set_title('Predicted Mask')
ax[1].axis('off')
ax[2].imshow(overlay)
ax[2].set_title('Overlay')
ax[2].axis('off')
plt.show()